<a href="https://colab.research.google.com/github/ksehar99/EyePACS-DL-Blockchain/blob/main/DR_DL_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2
from PIL import Image
import numpy as np

# ── Setup ──────────────────────────────────────────────────────────────────
BASE_DIR = "/content/drive/MyDrive/EyePACS/EyePACS"
TARGET_SIZE = (224, 224)   # MobileNetV2 ka required input size
BATCH_SIZE = 32

# ── Step A: Folder check ───────────────────
def check_data(base_dir):
    info = {}
    # Update the splits list to correctly handle 'test_merge' as a single directory
    splits_to_check = ["train", "valid"]
    test_split_path = os.path.join("test", "test_merge") # Path to the actual image directory

    for split_dir in splits_to_check: # For train and valid, we expect subfolders
        for cls in ["Nrdr", "Rdr"]:
            path = os.path.join(base_dir, split_dir, cls)
            if os.path.exists(path):
                files = os.listdir(path)
                info[f"{split_dir}/{cls}"] = len(files)
                if files:
                    sample = Image.open(os.path.join(path, files[0]))
                    print(f"{split_dir}/{cls}: {len(files)} images | sample size: {sample.size} | mode: {sample.mode}")
            else:
                print(f"{split_dir}/{cls}: PATH NOT FOUND")

    # Handle the 'test_merge' directory separately
    path = os.path.join(base_dir, test_split_path)
    if os.path.exists(path):
        files = os.listdir(path)
        # Filter for image files only
        image_files = [f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif'))]
        info[test_split_path] = len(image_files)
        if image_files:
            sample = Image.open(os.path.join(path, image_files[0]))
            print(f"{test_split_path}: {len(image_files)} images | sample size: {sample.size} | mode: {sample.mode} (Labels inferred from filenames)")
        else:
            print(f"{test_split_path}: 0 image files found.")
    else:
        print(f"{test_split_path}: PATH NOT FOUND")
    return info

print("Checking dataset...\n")
data_info = check_data(BASE_DIR)


Checking dataset...

train/Nrdr: 2100 images | sample size: (1024, 1024) | mode: RGB
train/Rdr: 2100 images | sample size: (1024, 936) | mode: RGB
valid/Nrdr: 600 images | sample size: (1024, 808) | mode: RGB
valid/Rdr: 600 images | sample size: (1024, 1024) | mode: RGB
test/test_merge: 600 images | sample size: (1024, 823) | mode: RGB (Labels inferred from filenames)


In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

BASE_DIR    = "/content/drive/MyDrive/EyePACS/EyePACS"
IMG_SIZE    = 224
BATCH_SIZE  = 32
EPOCHS_P1   = 10
EPOCHS_P2   = 10
SEED        = 42

In [ ]:
import os
import tensorflow as tf
import numpy as np

train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    os.path.join(BASE_DIR, "train"),
    labels="inferred",
    label_mode="binary",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

val_ds_raw = tf.keras.utils.image_dataset_from_directory(
    os.path.join(BASE_DIR, "valid"),
    labels="inferred",
    label_mode="binary",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Custom loading for test_ds_raw as labels are in filenames within test/test_merge
test_merge_path = os.path.join(BASE_DIR, "test", "test_merge")

image_paths = []
labels = []

# Collect image paths and infer labels from filenames
# Nrdr (0) for levels 0 and 1, Rdr (1) for levels 2, 3, 4
if os.path.exists(test_merge_path):
    for filename in os.listdir(test_merge_path):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
            full_path = os.path.join(test_merge_path, filename)
            image_paths.append(full_path)
            # Check for retinopathy level in filename to infer label
            if '_eyepacs_0_' in filename or '_eyepacs_1_' in filename: # No Retinopathy (Nrdr) or mild (level 1)
                labels.append(0)
            elif any(f'_eyepacs_{i}_' in filename for i in [2, 3, 4]): # Retinopathy (Rdr) levels 2, 3, 4
                labels.append(1)
            else:
                print(f"Warning: Could not infer label for {filename}. Assigning default 0.")
                labels.append(0) # Default to 0 for unknown
else:
    print(f"Error: Test merge directory not found at {test_merge_path}")

# Convert lists to TensorFlow tensors
image_paths_tf = tf.constant(image_paths)
labels_tf = tf.constant(labels, dtype=tf.float32) # `label_mode="binary"` expects float32

# Function to load and preprocess a single image
def load_and_preprocess_image(image_path, label):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3) # Assuming JPEG, adjust if necessary
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE)) # Resize to model's expected input size
    # No normalization here, as `prep_eval` will handle `preprocess_input` later
    return img, label

# Create the test dataset
test_ds_raw = tf.data.Dataset.from_tensor_slices((image_paths_tf, labels_tf))
test_ds_raw = test_ds_raw.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
test_ds_raw = test_ds_raw.batch(BATCH_SIZE)
test_ds_raw = test_ds_raw.cache().prefetch(tf.data.AUTOTUNE) # Add cache and prefetch for performance

class_names = ['Nrdr', 'Rdr'] # Explicitly define since not inferred from directory structure
print("Class mapping:", class_names)

Found 4200 files belonging to 2 classes.
Found 1200 files belonging to 2 classes.
Class mapping: ['Nrdr', 'Rdr']


In [ ]:
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

def prep_train(img, label):
    img = preprocess_input(img)
    return img, label

def prep_eval(img, label):
    img = preprocess_input(img)
    return img, label

print("Preparing training dataset...")
train_ds = train_ds_raw.map(prep_train, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
print("Training dataset prepared.")

print("Preparing validation dataset...")
val_ds   = val_ds_raw.map(prep_eval, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
print("Validation dataset prepared.")

print("Preparing test dataset...")
test_ds  = test_ds_raw.map(prep_eval, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
print("Test dataset prepared.")

all_labels = np.concatenate([y.numpy() for x, y in train_ds_raw], axis=0).flatten()
print("Class distribution:", np.bincount(all_labels.astype(int)))

Class distribution: [2100 2100]


In [ ]:
# def focal_loss(gamma=2.0, alpha=0.25):
#     def loss_fn(y_true, y_pred):
#         y_true = tf.cast(y_true, tf.float32)
#         epsilon = tf.keras.backend.epsilon()
#         y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)

#         pt = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
#         alpha_t = tf.where(tf.equal(y_true, 1), alpha, 1 - alpha)

#         fl = -alpha_t * tf.pow(1 - pt, gamma) * tf.math.log(pt)
#         return tf.reduce_mean(fl)
#     return loss_fn

In [ ]:
def build_model():
    base = MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights="imagenet"
    )
    base.trainable = False

    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = Model(inputs, outputs)
    return model, base

model, base_model = build_model()
model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,586,177 (9.87 MB)

 Trainable params: 328,193 (1.25 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

phase 1 retraining

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc"),
             tf.keras.metrics.Precision(name="precision"),
             tf.keras.metrics.Recall(name="recall")]
)

callbacks_p1 = [
    EarlyStopping(monitor="val_auc", patience=3, restore_best_weights=True, mode="max"),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
    ModelCheckpoint("best_model_p1.h5", monitor="val_auc", save_best_only=True, mode="max")
]

history_p1 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS_P1,
    callbacks=callbacks_p1
)

Epoch 1/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6898 - auc: 0.7516 - loss: 0.6202 - precision: 0.6955 - recall: 0.6627

132/132 ━━━━━━━━━━━━━━━━━━━━ 441s 3s/step - accuracy: 0.7271 - auc: 0.7944 - loss: 0.5550 - precision: 0.7397 - recall: 0.7010 - val_accuracy: 0.7700 - val_auc: 0.8431 - val_loss: 0.4876 - val_precision: 0.8068 - val_recall: 0.7100 - learning_rate: 0.0010
Epoch 2/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7552 - auc: 0.8301 - loss: 0.5003 - precision: 0.7676 - recall: 0.7253

132/132 ━━━━━━━━━━━━━━━━━━━━ 296s 2s/step - accuracy: 0.7600 - auc: 0.8364 - loss: 0.4926 - precision: 0.7800 - recall: 0.7243 - val_accuracy: 0.7717 - val_auc: 0.8550 - val_loss: 0.4826 - val_precision: 0.7485 - val_recall: 0.8183 - learning_rate: 0.0010
Epoch 3/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7644 - auc: 0.8444 - loss: 0.4818 - precision: 0.7704 - recall: 0.7424

132/132 ━━━━━━━━━━━━━━━━━━━━ 283s 2s/step - accuracy: 0.7743 - auc: 0.8544 - loss: 0.4672 - precision: 0.7942 - recall: 0.7405 - val_accuracy: 0.7725 - val_auc: 0.8557 - val_loss: 0.4860 - val_precision: 0.7519 - val_recall: 0.8133 - learning_rate: 0.0010
Epoch 4/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7707 - auc: 0.8541 - loss: 0.4706 - precision: 0.7856 - recall: 0.7423

132/132 ━━━━━━━━━━━━━━━━━━━━ 303s 2s/step - accuracy: 0.7771 - auc: 0.8572 - loss: 0.4644 - precision: 0.7997 - recall: 0.7395 - val_accuracy: 0.7892 - val_auc: 0.8593 - val_loss: 0.4637 - val_precision: 0.8028 - val_recall: 0.7667 - learning_rate: 0.0010
Epoch 5/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7925 - auc: 0.8656 - loss: 0.4532 - precision: 0.8125 - recall: 0.7515

132/132 ━━━━━━━━━━━━━━━━━━━━ 288s 2s/step - accuracy: 0.7976 - auc: 0.8705 - loss: 0.4456 - precision: 0.8138 - recall: 0.7719 - val_accuracy: 0.7783 - val_auc: 0.8610 - val_loss: 0.4784 - val_precision: 0.8848 - val_recall: 0.6400 - learning_rate: 0.0010
Epoch 6/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7748 - auc: 0.8525 - loss: 0.4732 - precision: 0.7972 - recall: 0.7260

132/132 ━━━━━━━━━━━━━━━━━━━━ 295s 2s/step - accuracy: 0.7862 - auc: 0.8653 - loss: 0.4529 - precision: 0.8088 - recall: 0.7495 - val_accuracy: 0.7833 - val_auc: 0.8654 - val_loss: 0.4607 - val_precision: 0.7760 - val_recall: 0.7967 - learning_rate: 0.0010
Epoch 7/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 287s 2s/step - accuracy: 0.8033 - auc: 0.8791 - loss: 0.4298 - precision: 0.8346 - recall: 0.7567 - val_accuracy: 0.7817 - val_auc: 0.8651 - val_loss: 0.4616 - val_precision: 0.7798 - val_recall: 0.7850 - learning_rate: 0.0010
Epoch 8/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7967 - auc: 0.8745 - loss: 0.4361 - precision: 0.8117 - recall: 0.7635

132/132 ━━━━━━━━━━━━━━━━━━━━ 337s 2s/step - accuracy: 0.8005 - auc: 0.8786 - loss: 0.4289 - precision: 0.8236 - recall: 0.7648 - val_accuracy: 0.7867 - val_auc: 0.8671 - val_loss: 0.4617 - val_precision: 0.8454 - val_recall: 0.7017 - learning_rate: 0.0010
Epoch 9/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8177 - auc: 0.8950 - loss: 0.4016 - precision: 0.8607 - recall: 0.7585

132/132 ━━━━━━━━━━━━━━━━━━━━ 289s 2s/step - accuracy: 0.8226 - auc: 0.9000 - loss: 0.3916 - precision: 0.8527 - recall: 0.7800 - val_accuracy: 0.7925 - val_auc: 0.8686 - val_loss: 0.4507 - val_precision: 0.8369 - val_recall: 0.7267 - learning_rate: 5.0000e-04
Epoch 10/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8145 - auc: 0.8959 - loss: 0.3961 - precision: 0.8357 - recall: 0.7750

132/132 ━━━━━━━━━━━━━━━━━━━━ 301s 2s/step - accuracy: 0.8210 - auc: 0.9036 - loss: 0.3856 - precision: 0.8460 - recall: 0.7848 - val_accuracy: 0.7917 - val_auc: 0.8700 - val_loss: 0.4501 - val_precision: 0.8017 - val_recall: 0.7750 - learning_rate: 5.0000e-04


phase 2 fine tuning

In [ ]:
# Load the best model from Phase 1 training
print("Loading best model from Phase 1 ('best_model_p1.h5')...")
loaded_model_p1 = tf.keras.models.load_model(
    'best_model_p1.h5',
    custom_objects={'BinaryCrossentropy': tf.keras.losses.BinaryCrossentropy}
)

# Re-assign model to the loaded one
model = loaded_model_p1

# We also need to get the base_model part from the loaded model
# Assuming base_model is the first layer of the actual functional model after InputLayer
# Find the layer that matches the MobileNetV2 architecture
for layer in model.layers:
    if isinstance(layer, tf.keras.Model) and layer.name == 'mobilenetv2':
        base_model = layer
        break

print("Model loaded for Phase 2 fine-tuning.")

Loading best model from Phase 1 ('best_model_p1.h5')...


Model loaded for Phase 2 fine-tuning.


In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc"),
             tf.keras.metrics.Precision(name="precision"),
             tf.keras.metrics.Recall(name="recall")]
)

callbacks_p2 = [
    EarlyStopping(monitor="val_auc", patience=5, restore_best_weights=True, mode="max"),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
    ModelCheckpoint("best_model_final.h5", monitor="val_auc", save_best_only=True, mode="max")
]

history_p2 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS_P2,
    callbacks=callbacks_p2
)

Epoch 1/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8332 - auc: 0.9098 - loss: 0.3715 - precision: 0.8513 - recall: 0.8015

132/132 ━━━━━━━━━━━━━━━━━━━━ 320s 2s/step - accuracy: 0.8371 - auc: 0.9165 - loss: 0.3622 - precision: 0.8583 - recall: 0.8076 - val_accuracy: 0.7900 - val_auc: 0.8706 - val_loss: 0.4473 - val_precision: 0.8096 - val_recall: 0.7583 - learning_rate: 1.0000e-05
Epoch 2/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8343 - auc: 0.9112 - loss: 0.3683 - precision: 0.8718 - recall: 0.7839

132/132 ━━━━━━━━━━━━━━━━━━━━ 281s 2s/step - accuracy: 0.8407 - auc: 0.9170 - loss: 0.3592 - precision: 0.8709 - recall: 0.8000 - val_accuracy: 0.7892 - val_auc: 0.8707 - val_loss: 0.4469 - val_precision: 0.8093 - val_recall: 0.7567 - learning_rate: 1.0000e-05
Epoch 3/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8317 - auc: 0.9147 - loss: 0.3614 - precision: 0.8642 - recall: 0.7810

132/132 ━━━━━━━━━━━━━━━━━━━━ 308s 2s/step - accuracy: 0.8419 - auc: 0.9195 - loss: 0.3550 - precision: 0.8709 - recall: 0.8029 - val_accuracy: 0.7900 - val_auc: 0.8709 - val_loss: 0.4469 - val_precision: 0.8118 - val_recall: 0.7550 - learning_rate: 1.0000e-05
Epoch 4/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8320 - auc: 0.9117 - loss: 0.3671 - precision: 0.8656 - recall: 0.7820

132/132 ━━━━━━━━━━━━━━━━━━━━ 314s 2s/step - accuracy: 0.8374 - auc: 0.9190 - loss: 0.3566 - precision: 0.8665 - recall: 0.7976 - val_accuracy: 0.7900 - val_auc: 0.8711 - val_loss: 0.4464 - val_precision: 0.8118 - val_recall: 0.7550 - learning_rate: 1.0000e-05
Epoch 5/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8357 - auc: 0.9132 - loss: 0.3657 - precision: 0.8745 - recall: 0.7803

132/132 ━━━━━━━━━━━━━━━━━━━━ 311s 2s/step - accuracy: 0.8398 - auc: 0.9198 - loss: 0.3544 - precision: 0.8706 - recall: 0.7981 - val_accuracy: 0.7900 - val_auc: 0.8714 - val_loss: 0.4461 - val_precision: 0.8107 - val_recall: 0.7567 - learning_rate: 1.0000e-05
Epoch 6/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8357 - auc: 0.9172 - loss: 0.3570 - precision: 0.8647 - recall: 0.7886

132/132 ━━━━━━━━━━━━━━━━━━━━ 334s 2s/step - accuracy: 0.8417 - auc: 0.9216 - loss: 0.3519 - precision: 0.8693 - recall: 0.8043 - val_accuracy: 0.7900 - val_auc: 0.8714 - val_loss: 0.4464 - val_precision: 0.8107 - val_recall: 0.7567 - learning_rate: 1.0000e-05
Epoch 7/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 279s 2s/step - accuracy: 0.8433 - auc: 0.9211 - loss: 0.3531 - precision: 0.8709 - recall: 0.8062 - val_accuracy: 0.7883 - val_auc: 0.8712 - val_loss: 0.4467 - val_precision: 0.8078 - val_recall: 0.7567 - learning_rate: 1.0000e-05
Epoch 8/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 322s 2s/step - accuracy: 0.8457 - auc: 0.9217 - loss: 0.3513 - precision: 0.8750 - recall: 0.8067 - val_accuracy: 0.7908 - val_auc: 0.8714 - val_loss: 0.4466 - val_precision: 0.8133 - val_recall: 0.7550 - learning_rate: 5.0000e-06
Epoch 9/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8381 - auc: 0.9159 - loss: 0.3591 - precision: 0.8681 - recall: 0.7930

132/132 ━━━━━━━━━━━━━━━━━━━━ 280s 2s/step - accuracy: 0.8476 - auc: 0.9223 - loss: 0.3494 - precision: 0.8751 - recall: 0.8110 - val_accuracy: 0.7908 - val_auc: 0.8715 - val_loss: 0.4468 - val_precision: 0.8133 - val_recall: 0.7550 - learning_rate: 5.0000e-06
Epoch 10/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 276s 2s/step - accuracy: 0.8486 - auc: 0.9236 - loss: 0.3488 - precision: 0.8769 - recall: 0.8110 - val_accuracy: 0.7917 - val_auc: 0.8714 - val_loss: 0.4467 - val_precision: 0.8147 - val_recall: 0.7550 - learning_rate: 2.5000e-06


test set evauation


In [ ]:
import pandas as pd

def tta_predict(model, img_batch, n_aug=4):
    """Average predictions over original + flipped versions"""
    preds = [model.predict(img_batch, verbose=0)]
    preds.append(model.predict(tf.image.flip_left_right(img_batch), verbose=0))
    preds.append(model.predict(tf.image.flip_up_down(img_batch), verbose=0))
    preds.append(model.predict(tf.image.flip_left_right(tf.image.flip_up_down(img_batch)), verbose=0))
    return np.mean(preds, axis=0)

y_true, y_pred_prob = [], []
for imgs, labels in test_ds:
    preds = tta_predict(model, imgs).flatten()
    y_pred_prob.extend(preds)
    y_true.extend(labels.numpy().flatten())

y_true = np.array(y_true)
y_pred_prob = np.array(y_pred_prob)
y_pred = (y_pred_prob >= 0.5).astype(int)

# ── Metrics table ───────────────────────────────────────────────────────
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

report_dict = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
metrics_df = pd.DataFrame(report_dict).transpose().round(4)
print(metrics_df)

print("\nOverall AUC:", round(roc_auc_score(y_true, y_pred_prob), 4))

cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm, index=[f"Actual_{c}" for c in class_names], columns=[f"Pred_{c}" for c in class_names])
print("\nConfusion Matrix:\n", cm_df)

              precision  recall  f1-score  support
Nrdr             0.7735  0.8767    0.8219   300.00
Rdr              0.8577  0.7433    0.7964   300.00
accuracy         0.8100  0.8100    0.8100     0.81
macro avg        0.8156  0.8100    0.8092   600.00
weighted avg     0.8156  0.8100    0.8092   600.00

Overall AUC: 0.8736

Confusion Matrix:
              Pred_Nrdr  Pred_Rdr
Actual_Nrdr        263        37
Actual_Rdr          77       223
